# Pipeline Medallion — `ecommerce_categorias`

Este notebook implementa a arquitetura **Medallion (Bronze → Silver → Gold)** para a tabela de categorias do e-commerce, utilizando **PySpark no Databricks** com armazenamento no **Azure Data Lake Storage Gen2 (ADLS)** e escrita no formato **Delta Lake**.

---

## 🗂️ Fluxo do Pipeline

```
RAW (CSV no ADLS)
      │
      ▼
  🥉 BRONZE  — Ingestão bruta com metadados de auditoria
      │
      ▼
  🥈 SILVER  — Limpeza de hierarquia, remoção de acentos, deduplicação
      │
      ▼
  🥇 GOLD    — KPI de diversificação (subcategorias por raiz)
      │
      ▼
  🗄️ SQL SERVER — Exportação do KPI para consumo em relatórios/BI
```

---

## 📐 Regras de Negócio Aplicadas (Silver)

| # | Regra | Descrição |
|---|-------|-----------|
| 1 | **Unicidade e deduplicação** | Remoção de registros com `id_categoria` nulo e eliminação de duplicatas pela mesma chave |
| 2 | **Tipagem e preservação de hierarquia** | Cast duplo (double → string) para preservar integridade numérica e permitir auto-join (`id_categoria_pai`) |
| 3 | **Limpeza de caracteres especiais** | Remoção de acentuação em `nome_categoria` para padronização alfanumérica |
| 4 | **Validação de referência** | Garantir que `id_categoria_pai` referencia uma categoria raiz válida (controle durante agregação Gold) |
| 5 | **Auditoria da camada** | Timestamp `silver_processed_at` para rastreamento de quando o registro foi processado |

---

## 📊 KPI Gerado (Gold)

| Tabela Gold | Descrição | Granularidade |
|-------------|-----------|---------------|
| `gold_kpi_subcategorias_por_raiz` | Contagem de subcategorias agrupadas por categoria raiz | Categoria Raiz |

---

## 🔧 Dependências e Pré-requisitos

- **Variáveis de ambiente** (arquivo `../env`): `CLIENT_ID`, `TENANT_ID`, `CLIENT_SECRET`, `STORAGE_ACCOUNT_NAME`, `SQL_HOST`, `SQL_DATABASE`, `SQL_USERNAME`, `SQL_PASSWORD`
- **Autenticação no ADLS**: OAuth 2.0 via Service Principal (Client Credentials)
- **Dataset SQL Server**: `squad3`

## 1. Setup e Credenciais

Carrega as variáveis de ambiente a partir do arquivo `../env` e configura:
- As opções de autenticação OAuth 2.0 para acesso ao ADLS Gen2 via Service Principal
- Os caminhos base das camadas **Raw** e **Bronze** no Data Lake
- Períodos de execução (`ANO_EXEC`, `MES_EXEC`, `DIA_EXEC`) para rastreamento semanal

> ⚠️ Nenhuma credencial é hardcoded. Todas as chaves sensíveis são lidas exclusivamente via `os.getenv()`.

In [0]:
# # ============================================================
# # 1. SETUP E CREDENCIAIS (ecommerce_categorias)
# # ============================================================
# import os
# from datetime import date
# from dotenv import load_dotenv
# from pyspark.sql.functions import current_timestamp, year, month, col, count, date_format, lit, ceil, lag, sum, round, avg, regexp_replace, when
# from pyspark.sql.window import Window

# load_dotenv("../env")
# client_id       = os.getenv("CLIENT_ID")
# tenant_id       = os.getenv("TENANT_ID")
# client_secret   = os.getenv("CLIENT_SECRET")
# storage_account = os.getenv("STORAGE_ACCOUNT_NAME")

# # Empacotando as credenciais em um dicionário de opções
# adls_options = {
#     "fs.azure.account.auth.type": "OAuth",
#     "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
#     "fs.azure.account.oauth2.client.id": client_id,
#     "fs.azure.account.oauth2.client.secret": client_secret,
#     "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
# }

# tabela      = "ecommerce_categorias"
# path_raw    = f"abfss://raw@{storage_account}.dfs.core.windows.net/batch-data/{tabela}.csv"
# path_bronze = f"abfss://squad3@{storage_account}.dfs.core.windows.net/bronze/{tabela}"

# # Período de referência da execução
# _hoje    = date.today()
# ANO_EXEC = _hoje.year
# MES_EXEC = _hoje.month
# DIA_EXEC = _hoje.day

# print(f"Configuração finalizada para a tabela: {tabela}")
# print(f"Período de execução: {DIA_EXEC}/{MES_EXEC}/{ANO_EXEC}")


In [0]:
# ============================================================
# 1. SETUP E CREDENCIAIS (ecommerce_categorias)
# ============================================================

import os
from datetime import date

from dotenv import load_dotenv

from pyspark.sql.functions import (
    current_timestamp,
    year,
    month,
    col,
    count,
    date_format,
    lit,
    ceil,
    lag,
    sum,
    round,
    avg,
    regexp_replace,
    when
)

from pyspark.sql.window import Window


# ============================================================
# 1.1 CARREGAMENTO DO .ENV + FALLBACK PARA JOB PARAMETERS
# ============================================================
# Objetivo:
# - Continuar funcionando manualmente com .env
# - Funcionar via Databricks Job usando Job Parameters
# - Evitar spark.conf.set(), pois no Serverless/Free Edition
#   algumas configs fs.azure.* não ficam disponíveis.
# ============================================================

try:
    load_dotenv("../env")
    load_dotenv("../.env")
    load_dotenv(".env")
    print("Tentativa de carregamento do .env realizada.")
except Exception as e:
    print(f"Não foi possível carregar .env. Seguindo com fallback. Detalhe: {e}")


def get_config_value(name: str, required: bool = True, default: str = "") -> str:
    """
    Busca uma configuração na seguinte ordem:

    1. Variáveis de ambiente carregadas pelo .env
    2. Databricks Job Parameters, via dbutils.widgets.get()
    3. Valor default, quando informado

    Isso permite que o notebook funcione tanto em execução manual
    quanto em execução agendada pelo Databricks Job.
    """

    value = os.getenv(name)

    if value is None or str(value).strip() == "":
        try:
            value = dbutils.widgets.get(name)
        except Exception:
            value = None

    if value is None or str(value).strip() == "":
        value = default

    value = str(value).strip() if value is not None else ""

    if required and value == "":
        raise ValueError(
            f"Configuração obrigatória não encontrada: {name}. "
            f"Verifique se ela existe no .env ou nos Job Parameters do Databricks."
        )

    return value


# ============================================================
# 1.2 VARIÁVEIS ADLS GEN2
# ============================================================

client_id = get_config_value("CLIENT_ID")
tenant_id = get_config_value("TENANT_ID")
client_secret = get_config_value("CLIENT_SECRET")
storage_account = get_config_value("STORAGE_ACCOUNT_NAME")

# Pelo seu path original, o arquivo está em:
# abfss://raw@storage/batch-data/ecommerce_categorias.csv
source_folder = get_config_value("CONTAINER_NAME", required=False, default="batch-data")

# Containers usados no seu projeto
raw_container = get_config_value("RAW_CONTAINER", required=False, default="raw")
squad_container = get_config_value("SQUAD_CONTAINER", required=False, default="squad3")


# ============================================================
# 1.3 OPÇÕES DE AUTENTICAÇÃO ADLS GEN2
# ============================================================
# Importante:
# NÃO usar spark.conf.set() aqui.
# No Databricks Free/Serverless, isso pode gerar:
# [CONFIG_NOT_AVAILABLE] Configuration fs.azure.* is not available.
#
# A estratégia é manter as opções em dicionário e usá-las nas leituras
# e escritas com .options(**adls_options).
# ============================================================

storage_account_fqdn = f"{storage_account}.dfs.core.windows.net"

adls_options = {
    f"fs.azure.account.auth.type.{storage_account_fqdn}": "OAuth",
    f"fs.azure.account.oauth.provider.type.{storage_account_fqdn}": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    f"fs.azure.account.oauth2.client.id.{storage_account_fqdn}": client_id,
    f"fs.azure.account.oauth2.client.secret.{storage_account_fqdn}": client_secret,
    f"fs.azure.account.oauth2.client.endpoint.{storage_account_fqdn}": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}

# Mantido também em formato genérico, caso alguma célula posterior
# já esteja usando esse dicionário antigo.
adls_options_generic = {
    "fs.azure.account.auth.type": "OAuth",
    "fs.azure.account.oauth.provider.type": "org.apache.hadoop.fs.azurebfs.oauth2.ClientCredsTokenProvider",
    "fs.azure.account.oauth2.client.id": client_id,
    "fs.azure.account.oauth2.client.secret": client_secret,
    "fs.azure.account.oauth2.client.endpoint": f"https://login.microsoftonline.com/{tenant_id}/oauth2/token"
}


# ============================================================
# 1.4 PATHS DA TABELA
# ============================================================

tabela = "ecommerce_categorias"

path_raw = (
    f"abfss://{raw_container}@{storage_account}.dfs.core.windows.net/"
    f"{source_folder}/{tabela}.csv"
)

path_bronze = (
    f"abfss://{squad_container}@{storage_account}.dfs.core.windows.net/"
    f"bronze/{tabela}"
)


# ============================================================
# 1.5 PERÍODO DE REFERÊNCIA DA EXECUÇÃO
# ============================================================

_hoje = date.today()
ANO_EXEC = _hoje.year
MES_EXEC = _hoje.month
DIA_EXEC = _hoje.day


# ============================================================
# 1.6 LOG SEGURO DE VALIDAÇÃO
# ============================================================

print(f"Configuração finalizada para a tabela: {tabela}")
print(f"Período de execução: {DIA_EXEC}/{MES_EXEC}/{ANO_EXEC}")
print(f"Storage Account: {storage_account}")
print(f"Raw Container: {raw_container}")
print(f"Source Folder: {source_folder}")
print(f"Squad Container: {squad_container}")
print(f"Path Raw: {path_raw}")
print(f"Path Bronze: {path_bronze}")
print("Credenciais carregadas sem expor secrets.")

## 2. 🥉 Camada Bronze — Ingestão RAW → Bronze

Realiza a ingestão do arquivo CSV bruto da camada Raw para a camada Bronze, **sem transformações de negócio** — apenas acréscimo de metadados de rastreabilidade:

| Coluna adicionada | Descrição |
|-------------------|-----------|
| `bronze_ingested_at` | Timestamp do momento da ingestão |
| `bronze_source_file` | Caminho completo do arquivo de origem (via `_metadata.file_path`) |
| `ano_particao` | Ano extraído do timestamp de ingestão — usado para particionamento físico |
| `mes_particao` | Mês extraído do timestamp de ingestão — usado para particionamento físico |

> 📌 O schema é lido como `string` (`inferSchema=false`) para preservar os dados exatamente como estão na fonte, evitando casting automático incorreto. A tipagem adequada ocorre na camada Silver.
>
> 📌 O modo de escrita é `append`, permitindo múltiplas execuções incrementais sem sobrescrever cargas anteriores.

In [0]:
# ============================================================
# 2. EXTRAÇÃO E CARGA (RAW -> BRONZE)
# ============================================================
print(f"Lendo {tabela} da camada Raw...")

# 1. LEITURA DA RAW
print(f"Lendo dados da Raw com inferSchema=false...")
df_raw = (
    spark.read
    .format("csv")
    .option("header", "true")
    .option("inferSchema", "false") # Mantendo tudo como String
    .options(**adls_options) 
    .load(path_raw)
)

print(f"Total de registros encontrados: {df_raw.count()}")

# 2. ADICIONANDO AS COLUNAS DE PARTIÇÃO E AUDITORIA
df_bronze = (
    df_raw
    .withColumn("bronze_ingested_at", current_timestamp())
    .withColumn("bronze_source_file", col("_metadata.file_path"))
    .withColumn("ano_particao", year(col("bronze_ingested_at")))
    .withColumn("mes_particao", month(col("bronze_ingested_at")))
)

print(f"Gravando fisicamente na camada Bronze: {path_bronze}")

print(f"Iniciando gravação física na camada Bronze...")

# 3. CARGA FÍSICA (WRITE BRONZE) - CÓDIGO DE PRODUÇÃO
print(f"Iniciando gravação incremental na camada Bronze...")

(
    df_bronze.write
    .format("delta")
    .mode("append") # <--- Voltamos para o comportamento incremental
    .option("mergeSchema", "true") # <--- Proteção extra: se a origem enviar uma coluna nova no futuro, o Delta aceita sem quebrar
    .options(**adls_options) 
    .partitionBy("ano_particao", "mes_particao")
    .save(path_bronze)
)

print(f"SUCESSO! Ingestão incremental concluída com segurança.")

print(f"SUCESSO! O passado foi apagado e a Bronze agora tem as partições e é 100% String.")

print("Ingestão Bronze finalizada com sucesso!")
display(df_bronze.limit(5))

In [0]:

# ============================================================
# 3. CAMADA SILVER (CATEGORIAS - LIMPEZA E HIERARQUIA)
# ============================================================
from pyspark.sql.functions import current_timestamp, col, translate

print(f"Iniciando processamento da camada Silver para: {tabela}...")

# Define o caminho de destino na Silver
path_silver = f"abfss://squad3@{storage_account}.dfs.core.windows.net/silver/{tabela}"

# Lendo a tabela Delta da Bronze
df_bronze_read = (
    spark.read
    .format("delta")
    .options(**adls_options) 
    .load(path_bronze)
)

# Mapeamento para remoção de acentos (Regra 2)
com_acentos = "áàâãäéèêëíìîïóòôõöúùûüçÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ"
sem_acentos = "aaaaaeeeeiiiiooooouuuucAAAAAEEEEIIIIOOOOOUUUUC"

df_silver = (
    df_bronze_read
    
    # 1. Unicidade Básica
    .dropna(subset=["id_categoria"])
    .dropDuplicates(["id_categoria"])
    
    # 2. Tipagem e Preservação de Hierarquia (O TRUQUE DO CAST DUPLO ESTÁ AQUI)
    .withColumn("id_categoria", col("id_categoria").cast("double").cast("string"))
    .withColumn("id_categoria_pai", col("id_categoria_pai").cast("double").cast("string")) 
    
    # 3. Limpeza de Caracteres Especiais
    .withColumn("nome_categoria", translate(col("nome_categoria").cast("string"), com_acentos, sem_acentos))
    
    # 4. Auditoria da camada Silver
    .withColumn("silver_processed_at", current_timestamp())
)


# ============================================================
# 4. CARGA FÍSICA (WRITE SILVER)
# ============================================================
print(f"Gravando dados limpos fisicamente em: {path_silver}")

(
    df_silver.write
    .format("delta")
    .mode("append") 
    .option("mergeSchema", "true") 
    .options(**adls_options) 
    .partitionBy("ano_particao", "mes_particao")
    .save(path_silver)
)

print("SUCESSO! Categoria refinada e sem acentos salva na Silver.\n")

display(df_silver.limit(5))

## 3. 🥈 Camada Silver — Limpeza, Deduplicação e Preservação de Hierarquia

Esta é a etapa central de transformação do pipeline. A Silver recebe os dados brutos da Bronze e aplica todas as regras de qualidade e negócio, com foco especial na **preservação da hierarquia de categorias** (categoria raiz ↔ subcategoria).

### Estratégia de Tipagem

A coluna `id_categoria_pai` é crítica para manter a integridade relacional. O pipeline usa um **cast duplo**:
```python
col("id_categoria_pai").cast("double").cast("string")
```

Esse padrão garante que:
1. **Double**: Remove leading zeros e valida numericamente (ex: "007" → 7.0)
2. **String**: Retorna ao formato string compatível com join (ex: 7.0 → "7")

### Schema de saída da Silver

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `id_categoria` | String | Chave primária (categoria ou subcategoria) |
| `id_categoria_pai` | String | Referência para categoria pai (null = raiz) |
| `nome_categoria` | String | Nome da categoria sem acentos |
| `ano_particao` | Integer | Ano da ingestão (partição física) |
| `mes_particao` | Integer | Mês da ingestão (partição física) |
| `silver_processed_at` | Timestamp | Timestamp de processamento para auditoria |

In [0]:
# ============================================================
# 4. CAMADA GOLD (KPI: DIVERSIFICAÇÃO DO CATÁLOGO)
# ============================================================
from pyspark.sql.functions import col, count, current_timestamp, year, month, date_format, lit

print("Iniciando processamento da camada Gold (Qtd de Subcategorias por Raiz)...")

tabela_gold = "gold_kpi_subcategorias_por_raiz"
path_gold_agg = f"abfss://squad3@{storage_account}.dfs.core.windows.net/gold/{tabela_gold}"

# Releitura da Silver
df_silver = spark.read.format("delta").options(**adls_options).load(path_silver)

# 1. Filtramos quem é Categoria Raiz (não tem id_categoria_pai)
df_raiz = (
    df_silver
    .filter(col("id_categoria_pai").isNull() | (col("id_categoria_pai") == "null"))
    .select(
        col("id_categoria").alias("id_categoria_raiz"),
        col("nome_categoria").alias("nome_categoria_raiz")
    )
)

# 2. Filtramos quem é Subcategoria (tem id_categoria_pai preenchido)
df_subcategorias = (
    df_silver
    .filter(col("id_categoria_pai").isNotNull() & (col("id_categoria_pai") != "null"))
    .select(
        col("id_categoria").alias("id_subcategoria"),
        col("id_categoria_pai")
    )
)

# 3. Cruzamento e Cálculo do KPI
df_gold_agg = (
    df_subcategorias
    .join(
        df_raiz,
        df_subcategorias.id_categoria_pai == df_raiz.id_categoria_raiz,
        "inner"
    )
    .groupBy("id_categoria_raiz", "nome_categoria_raiz")
    .agg(
        count("id_subcategoria").alias("qtd_subcategorias")
    )
    .withColumn("data_fotografia", current_timestamp())
    .withColumn("ano_particao", year(col("data_fotografia")))
    .withColumn("mes_particao", month(col("data_fotografia")))
    .withColumn("dia_exec", lit(DIA_EXEC))
    .withColumn("gold_processed_at", date_format(current_timestamp(), "yyyy-MM-dd HH:mm:ss"))
)

print("Gravando Data Mart na camada Gold (MODO: DELETE + APPEND)...")

# Delete + Append com granularidade de dia
try:
    (
        spark.read
        .format("delta")
        .options(**adls_options)
        .load(path_gold_agg)
        .createOrReplaceTempView(f"tmp_{tabela_gold}")
    )
    spark.sql(f"""
        DELETE FROM delta.`{path_gold_agg}`
        WHERE ano_particao = {ANO_EXEC}
          AND mes_particao = {MES_EXEC}
          AND dia_exec     = {DIA_EXEC}
    """)
    print(f"Delete executado: {tabela_gold} [{DIA_EXEC}/{MES_EXEC}/{ANO_EXEC}]")
except Exception:
    print(f"Primeira carga detectada — sem delete: {tabela_gold}")

# Append
(
    df_gold_agg.write
    .format("delta")
    .mode("append")
    .option("mergeSchema", "true")
    .options(**adls_options)
    .partitionBy("ano_particao", "mes_particao")
    .save(path_gold_agg)
)

print("SUCESSO! KPI de diversificação de categorias gravado na Gold.\n")
display(df_gold_agg.limit(5))


In [0]:
# ============================================================
# 5. EXPORTAÇÃO PARA O SQL SERVER (SERVING LAYER)
# ============================================================
import os
from dotenv import load_dotenv

print("Iniciando a exportação do KPI para o SQL Server (MODO APPEND)...")

load_dotenv("../env")

jdbc_hostname = os.getenv("SQL_HOST")
jdbc_port = "1433"
jdbc_database = os.getenv("SQL_DATABASE")
jdbc_username = os.getenv("SQL_USERNAME")
jdbc_password = os.getenv("SQL_PASSWORD")

# Dataset squad3 — padrão do projeto
tabela_sql_server = "squad3.gold_kpi_subcategorias_por_raiz"

try:
    (
        df_gold_agg.write
        .format("sqlserver")
        .option("host", jdbc_hostname)
        .option("port", jdbc_port)
        .option("database", jdbc_database)
        .option("dbtable", tabela_sql_server)
        .option("user", jdbc_username)
        .option("password", jdbc_password)
        .mode("append")
        .save()
    )
    print(f"SUCESSO! Dados exportados para {tabela_sql_server} no SQL Server.")
except Exception as e:
    print(f"Erro ao exportar para o SQL Server:\n{e}")


## 4. 🥇 Camada Gold — Data Mart de Diversificação de Catálogo

A camada Gold consome a Silver e gera um **KPI de diversificação**, contando quantas subcategorias pertencem a cada categoria raiz. Este Data Mart segue princípios de **Star Schema**, com categorias raiz como dimensão e subcategorias como métrica.

### Estratégia de Cálculo

1. **Filtrar Raízes**: Identificar categorias com `id_categoria_pai` nulo (raiz)
2. **Filtrar Subcategorias**: Identificar categorias com `id_categoria_pai` preenchido
3. **Join Relacional**: Cruzar subcategorias com suas raízes
4. **Agregação**: Contar subcategorias por raiz
5. **Idempotência**: Delete + append com granularidade de dia de execução

### KPI Gerado

| Coluna | Tipo | Descrição |
|--------|------|-----------|
| `id_categoria_raiz` | String | Chave primária da categoria raiz |
| `nome_categoria_raiz` | String | Nome descritivo da raiz |
| `qtd_subcategorias` | Long | Quantidade de subcategorias |
| `data_fotografia` | Timestamp | Momento em que o KPI foi calculado |
| `ano_particao` | Integer | Ano (partição física) |
| `mes_particao` | Integer | Mês (partição física) |
| `dia_exec` | Integer | Dia da execução (controle de idempotência) |
| `gold_processed_at` | String | Timestamp de processamento |

> 📌 A escrita Gold utiliza **delete + append por dia de execução** (`dia_exec`), garantindo idempotência em reprocessamentos da mesma semana sem apagar cargas de semanas anteriores do mesmo mês.

## 6. 📊 Data Quality — Sanidade Analítica (Hierarquia de Categorias)

Análise gráfica e textual da qualidade dos dados em cada etapa de transformação, monitorando os 5 problemas mais comuns na validação de categorias e sua hierarquia.

## 5. 🗄️ Exportação para SQL Server

Etapa final do pipeline: o KPI calculado na camada Gold é **exportado para o SQL Server** no schema `squad3`, tornando-o disponível para consumo por ferramentas de BI e relatórios.

### Estratégia de carga

- **Modo:** `append` — novos registros são inseridos sem sobrescrever o histórico existente
- **Conector:** driver nativo `sqlserver` do Spark
- **Schema:** `squad3` (padrão do projeto)
- **Tabela de destino:** `squad3.gold_kpi_subcategorias_por_raiz`

> ⚠️ As credenciais do SQL Server (`SQL_HOST`, `SQL_DATABASE`, `SQL_USERNAME`, `SQL_PASSWORD`) são carregadas via `load_dotenv()` e nunca expostas no código.

In [0]:
# ============================================================
# 6. DATA QUALITY - MONITORAMENTO DE HIERARQUIA DE CATEGORIAS
# ============================================================
from pyspark.sql.functions import count, col as col_func
from builtins import round as python_round
import matplotlib.pyplot as plt
import pandas as pd

print("Gerando insights de qualidade da Silver (Categorias)...\n")

# Releitura da Bronze e Silver para comparação
df_bronze_check = spark.read.format("delta").options(**adls_options).load(path_bronze)
df_silver_check = spark.read.format("delta").options(**adls_options).load(path_silver)
total_registros_bronze = df_bronze_check.count()
total_registros_silver = df_silver_check.count()

# REGRA 1: Nulos em id_categoria (Bronze)
regra_1_erros = df_bronze_check.filter(col_func("id_categoria").isNull()).count()

# REGRA 2: Duplicatas em id_categoria (Bronze)
regra_2_erros = total_registros_bronze - df_bronze_check.dropDuplicates(["id_categoria"]).count()

# REGRA 3: Acentuação RESIDUAL na Silver (deve ser 0 se o translate funcionou)
acentos = "áàâãäéèêëíìîïóòôõöúùûüçÁÀÂÃÄÉÈÊËÍÌÎÏÓÒÔÕÖÚÙÛÜÇ"
com_acentos = 0
for char in acentos:
    com_acentos += df_silver_check.filter(col_func("nome_categoria").contains(char)).count()
regra_3_erros = com_acentos

# REGRA 4: id_categoria_pai inválido (roda contra SILVER com tipos já tratados)
df_raiz_check = df_silver_check.filter(
    col_func("id_categoria_pai").isNull() | (col_func("id_categoria_pai") == "null")
).select(col_func("id_categoria").alias("id_raiz"))

pai_orfao = (
    df_silver_check
    .filter(col_func("id_categoria_pai").isNotNull() & (col_func("id_categoria_pai") != "null"))
    .join(df_raiz_check, col_func("id_categoria_pai") == col_func("id_raiz"), "left")
    .filter(col_func("id_raiz").isNull())
    .count()
)
regra_4_erros = pai_orfao

# REGRA 5: Auto-referência (Silver)
regra_5_erros = df_silver_check.filter(col_func("id_categoria") == col_func("id_categoria_pai")).count()

# ============================================================
# Montar DataFrame com os resultados
# ============================================================

dados_qualidade = {
    'Regra': [
        'R1: id_categoria\nNULO (Bronze)',
        'R2: Duplicatas\n(Bronze)',
        'R3: Acentos\nResiduais (Silver)',
        'R4: Categoria pai\nÓrfã (Silver)',
        'R5: Auto-\nReferência (Silver)'
    ],
    'Erros Encontrados': [
        regra_1_erros,
        regra_2_erros,
        regra_3_erros,
        regra_4_erros,
        regra_5_erros
    ],
    'Taxa Erro (%)': [
        python_round(100 * regra_1_erros / total_registros_bronze, 2) if total_registros_bronze > 0 else 0,
        python_round(100 * regra_2_erros / total_registros_bronze, 2) if total_registros_bronze > 0 else 0,
        python_round(100 * regra_3_erros / total_registros_silver, 2) if total_registros_silver > 0 else 0,
        python_round(100 * regra_4_erros / total_registros_silver, 2) if total_registros_silver > 0 else 0,
        python_round(100 * regra_5_erros / total_registros_silver, 2) if total_registros_silver > 0 else 0,
    ]
}

df_qualidade = pd.DataFrame(dados_qualidade)

# ============================================================
# Exibir Tabela Resumo
# ============================================================

print(f"\n{'='*70}")
print(f"ANÁLISE DE QUALIDADE - REGRAS SILVER (CATEGORIAS)")
print(f"{'='*70}")
print(f"Total de registros na Bronze: {total_registros_bronze:,}")
print(f"Total de registros na Silver: {total_registros_silver:,}")
print(f"Duplicatas removidas: {total_registros_bronze - total_registros_silver:,}")
print(f"Taxa de rejeição geral: {python_round(100 * (total_registros_bronze - total_registros_silver) / total_registros_bronze, 2)}%\n")
print(df_qualidade.to_string(index=False))
print(f"{'='*70}\n")

# ============================================================
# Gráficos
# ============================================================

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

colors_gradient = ['#ff6b6b', '#ff8c8c', '#ffa9a9', '#ffc6c6', '#ffe0e0']
bars1 = ax1.bar(range(len(df_qualidade)), df_qualidade['Erros Encontrados'], color=colors_gradient, edgecolor='#333', linewidth=1.5)
ax1.set_ylabel('Quantidade de Registros com Erro', fontsize=11, fontweight='bold')
ax1.set_title('Erros Detectados por Regra de Validação', fontsize=12, fontweight='bold')
ax1.set_xticks(range(len(df_qualidade)))
ax1.set_xticklabels(df_qualidade['Regra'], fontsize=9)
ax1.grid(axis='y', alpha=0.3, linestyle='--')

for bar in bars1:
    height = bar.get_height()
    if height > 0:
        ax1.text(bar.get_x() + bar.get_width()/2., height,
                f'{int(height):,}',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

colors_taxa = ['#e74c3c' if x > 5 else '#f39c12' if x > 1 else '#27ae60' for x in df_qualidade['Taxa Erro (%)']]
bars2 = ax2.bar(range(len(df_qualidade)), df_qualidade['Taxa Erro (%)'], color=colors_taxa, edgecolor='#333', linewidth=1.5)
ax2.set_ylabel('Taxa de Erro (%)', fontsize=11, fontweight='bold')
ax2.set_title('Taxa de Erro por Regra (% do Total)', fontsize=12, fontweight='bold')
ax2.set_xticks(range(len(df_qualidade)))
ax2.set_xticklabels(df_qualidade['Regra'], fontsize=9)
ax2.grid(axis='y', alpha=0.3, linestyle='--')
ax2.axhline(y=1, color='#f39c12', linestyle='--', linewidth=1, alpha=0.5, label='Limite Aceitável (1%)')

for bar in bars2:
    height = bar.get_height()
    if height > 0:
        ax2.text(bar.get_x() + bar.get_width()/2., height,
                f'{height:.2f}%',
                ha='center', va='bottom', fontsize=9, fontweight='bold')

plt.tight_layout()
plt.show()

print("✅ Insights de qualidade gerados com sucesso.\n")

In [0]:
# # ============================================================
# # TRUNCATE (executar UMA VEZ para limpar duplicatas acumuladas)
# # ============================================================

# path_bronze_trunc = f"abfss://squad3@{storage_account}.dfs.core.windows.net/bronze/{tabela}"
# path_silver_trunc = f"abfss://squad3@{storage_account}.dfs.core.windows.net/silver/{tabela}"
# path_gold_trunc   = f"abfss://squad3@{storage_account}.dfs.core.windows.net/gold/gold_kpi_subcategorias_por_raiz"

# for path in [path_bronze_trunc, path_silver_trunc, path_gold_trunc]:
#     try:
#         df_vazio = spark.read.format("delta").options(**adls_options).load(path).limit(0)
#         df_vazio.write.format("delta").mode("overwrite").options(**adls_options).save(path)
#         print(f"✅ Truncado: {path.split('/')[-1]}")
#     except Exception:
#         print(f"⏭️ Tabela não encontrada (skip): {path.split('/')[-1]}")

# print("\nPronto! Re-execute o pipeline do início.")